In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
hojjatk_mnist_dataset_path = kagglehub.dataset_download('hojjatk/mnist-dataset')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import datasets
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader

In [ ]:
#first step - Define transformations for images

    #ToTensor :-> Not only does image -> Tensor, but scales by dividing values by 255 hence range of 0-1
Tensor = transforms.ToTensor()

MNIST = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=Tensor
)

loader = DataLoader(MNIST, batch_size=64, shuffle=False)
print('done')

In [ ]:
#finding the mean and std
#Why?: To normalise

sum_pixels = 0.0
sum_squared = 0.0
num_pixels = 0
for images, _ in loader:
    sum_pixels += images.sum()
    sum_squared += (images ** 2).sum()
    num_pixels += images.numel()

mean = sum_pixels / num_pixels
variance = (sum_squared / num_pixels) - (mean ** 2)
std = torch.sqrt(variance)

print("Mean:", mean.item())
print("Std:", std.item())

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((mean.item(),), (std.item(),))
])
train_dataset = torchvision.datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)
test_dataset = torchvision.datasets.MNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [ ]:
#VISUALISE
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.pyplot as plt
# Function to display images
def show_images(images, labels):
    fig, axes = plt.subplots(2, 5, figsize=(10, 4))
    axes = axes.flatten()
    for i in range(10):
        axes[i].imshow(images[i].reshape(28, 28), cmap='Oranges')
        axes[i].set_title(labels[i])
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()

# Get some random training images

images, labels = next(iter(train_loader))

# Show images
show_images(images[:10], labels[:10])

In [ ]:
#building the model

class MNISTNN(nn.Module):
    def __init__(self):
        super().__init__()
        #Flatten images to be one long vector
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28*28,128)
        self.relu = nn.ReLU()
        self.fc2=nn.Linear(128,64)
        self.fc3=nn.Linear(64,10)
    def forward(self,x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)
        return x

#initialise the model
model = MNISTNN()
print(model)



In [ ]:
#Training
#forward,loss,zero gradients, backpropagation, gradient descent

#define loss and optimizer
#for MNIST dataset we use the sparse Categorical Cross Entropyloss, in pytorch its "cross entropy loss"
criterion = nn.CrossEntropyLoss()
#via YonatanBest on Github, AdaGrad was shown to have best general performance in images
optimise = optim.Adagrad(model.parameters(),lr = 0.1)

#Training Loop
epochs = 10
train_losses = []
for epoch in range(epochs):
    running_loss=0.0
#MNIST has thousands of images so we take in batches
for i, (input,labels) in enumerate(train_loader):
#zero grad only needs to happen before backpropagation
    optimise.zero_grad()
#forward pass (inputs -> X_train)
    output = model(input)
#loss (outputs -> y_pred, labels -> y_train)
    loss = criterion(output, labels)
#backpropagation
    loss.backward()
#progress optimiser
    optimise.step()

  # Print statistics
    running_loss += loss.item()
    if (i+1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Step [{i+1}/{len(train_loader)}], Loss: {running_loss/100:.4f}')
        train_losses.append(running_loss/100)
        running_loss = 0.0

print('Training finished!')
plt.figure(figsize=(10, 5))
plt.plot(train_losses)
plt.title('Training Loss')
plt.xlabel('Training Steps (x100)')
plt.ylabel('Loss')
plt.grid(True)
plt.show()


In [ ]:
#Evaluate Model
model.eval()
correct = 0
total = 0
with torch.inference_mode():
    for images,labels in test_loader:
        outputs = model(images)
        #this random ah underscore is quite literally a python trash bin :)
        #so, we just need the predicted not the details like values so BOOM TRASH
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
accuracy = 100 * correct / total
print(f'Accuracy on the test set: {accuracy:.2f}%')

In [ ]:
def show_predictions(images, labels, preds):
    fig, axes = plt.subplots(2, 5, figsize=(10, 4))
    axes = axes.flatten()
    for i in range(10):
        axes[i].imshow(images[i].reshape(28, 28), cmap='gray')
        color = 'green' if preds[i] == labels[i] else 'red'
        axes[i].set_title(f"Pred: {preds[i]}, True: {labels[i]}", color=color)
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()
# Get predictions for some test images
dataiter = iter(test_loader)
images, labels = next(dataiter)

outputs = model(images)
_, preds = torch.max(outputs, 1)

# Show predictions
show_predictions(images[:10], labels[:10], preds[:10])
#ALL OF THIS IS JUST A MLP (help)

In [ ]:
# Save the model
torch.save(model.state_dict(), 'mnist_model.pth')
print("Model saved!")

# Load the model (for future use)
loaded_model = MNISTNN()
loaded_model.load_state_dict(torch.load('mnist_model.pth'))
loaded_model.eval()
print("Model loaded")

In [ ]:
#Now a proper CNN
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()
    #first we use a Convultional layer, 2d is most common for image classification
    #input channel=1 (due to grayscale shape [1,28,28]) if it was RGB -> 3
    #output=32 (most common to do 16,32,64,128)
    #kernel size=3*3 why? its the most common used in CNNs however larger computation 5*5 and 7*7
    #stride=1 goes pixel by pixel and captures by detail, (28*28 is already small)
    #padding =1 -> to prevent loss of shape from 1,28,28 to 1,26,26
        self.conv1 = nn.Conv2d(1,32, kernel_size =3, stride =1, padding =1)
        self.relu = nn.ReLU()
        #max pooling takes an area and takes the largest value, this is for generalisation and reducing comp.
        #28*28 ->> 14*14
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        #why conv again? image has gotten more abstract so output size must increase for complexity
        self.conv2 = nn.Conv2d(32,64, kernel_size=3, stride=1, padding=1)
        #y=xW^t+b , linear is for fully connecting, combines layers
        #why 64*7*7, the shape after conv1->pool->conv2->pool will be 64*7*7 so now
        #why 128? we must reduce the product of 64*7*7, 128 is just a common hyperparemeter
        #you can use 256,512,64 etc
        self.fc1 = nn.Linear(64*7*7,128)
        #why 10? there are 10 classes 0-9
        self.fc2 = nn.Linear(128,10)
#whats forward? -> the mathematical operation that image goes through
    def forward(self,x):
        #extracts edges,textures, sees more than raw pixels
        x=self.conv1(x)
        #introduces nonlinearity, keep activations strong and useful (complex recognition, deep learning)
        x=self.relu(x)
        #now that we have good activations, we can afford to compress them
        x=self.pool(x)
        #second relay just repeats but now recognizes bigger shapes/structures
        x=self.conv2(x)
        x=self.relu(x)
        x=self.pool(x)
        #this finds batch size (via -1), and reshapes tensor, we do this since we still have a 3d feature map
        #for linear layers we obviously cant use 3d
        x=x.view(-1,64*7*7)
        #feature compression
        x=self.fc1(x)
        x=self.relu(x)
        #final prediction from 0-9 range
        x=self.fc2(x)
        return x


In [ ]:

criterion = nn.CrossEntropyLoss()
model = CNN()
optimise = optim.Adagrad(model.parameters(),lr = 0.1)
#Training Loop
epochs = 10
train_losses = []
for epoch in range(epochs):
    running_loss=0.0
    for i, (inputs,labels) in enumerate(train_loader):
        optimise.zero_grad()
#forward pass (inputs -> X_train)
        output = model(inputs)
#loss (outputs -> y_pred, labels -> y_train)
        loss = criterion(output, labels)
#backpropagation
        loss.backward()
#progress optimiser
        optimise.step()

  # Print statistics
        running_loss += loss.item()
        if (i+1) % 100== 0:
            print(f'Epoch [{epoch+1}/{epochs}], Step [{i+1}/{len(train_loader)}], Loss: {running_loss/100:.4f}')
            train_losses.append(running_loss/100)
            running_loss = 0.0

print('Training finished!')
plt.figure(figsize=(10, 5))
plt.plot(train_losses)
plt.title('Training Loss')
plt.xlabel('Training Steps (x100)')
plt.ylabel('Loss')
plt.grid(True)
plt.show()